# Multiple Linear Regression from Scratch — Diabetes Blood Pressure Prediction

This notebook implements Multiple Linear Regression using the **Normal Equation**
(closed-form solution, no gradient descent) built from scratch in `MultiLinearRegression.py`,
and validates it against scikit-learn's `LinearRegression`.

**Goal:** predict blood pressure (`bp`) from multiple patient features, and compare against
the single-feature (BMI-only) regression baseline from the earlier SLR project (R² ≈ 0.233).

In [ ]:
import pandas as py
import numpy as np
import matplotlib.pyplot as mp
import seaborn as sns
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

from MultiLinearRegression import MultiLinearRegression

   Size(x1)  Bedroom(x2)
0         4            2
1         6            3
2         8            1
0    19
1    26
2    24
Name: Price(y), dtype: int64
[[3214.]
 [4617.]
 [6020.]]


## Load & Prepare Data

Using sklearn's built-in diabetes dataset. Columns are renamed from their generic `s1`–`s6`
labels to their real clinical meaning for readability.

In [ ]:
diabetes = load_diabetes()
df = py.DataFrame(
    data=diabetes.data,
    columns=diabetes.feature_names
)
df['target'] = diabetes.target
df = df.rename(columns={
    "s1": "total_cholesterol",
    "s2": "ldl",
    "s3": "hdl",
    "s4": "tc_hdl_ratio",
    "s5": "triglycerides",
    "s6": "glucose"
})
df.head()

,age,sex,bmi,bp,total_cholesterol,ldl,hdl,tc_hdl_ratio,triglycerides,glucose,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


## Explore Feature Correlations with Blood Pressure

`target` (the dataset's actual disease-progression score) is excluded here — it's the
*label* for a different prediction task, not something you'd have on hand when predicting
a patient's blood pressure in practice. Using it as a feature would be data leakage.

In [ ]:
cor = df.drop(columns=["target"]).corr(numeric_only=True)
cor['bp'].sort_values(ascending=True)

hdl                 -0.178762
ldl                  0.185548
sex                  0.241010
total_cholesterol    0.242464
tc_hdl_ratio         0.257650
age                  0.335428
glucose              0.390430
triglycerides        0.393480
bmi                  0.395411
bp                   1.000000
Name: bp, dtype: float64

## Feature Selection & Train/Test Split

Selected `bmi`, `triglycerides`, `glucose`, and `age` — the four real, clinically
available features with the strongest correlation to `bp`.

Using an 80/20 train/test split (`test_size=0.2`) to match the SLR and Gradient Descent
projects, so R² is directly comparable across all three.

In [ ]:
X = df[['bmi', 'triglycerides', 'glucose', 'age']]
y = df['bp']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)
X_train.shape, X_test.shape

((353, 4), (89, 4))

## Fit Multiple Linear Regression (Normal Equation, from scratch)

`MultiLinearRegression` solves for all coefficients in one closed-form step:
θ = (XᵀX)⁻¹Xᵀy — no learning rate or iterations needed, unlike Gradient Descent.

In [ ]:
model = MultiLinearRegression()
model.fit(X_train, y_train)

## Predict on the Test Set

In [ ]:
y_pred = model.predict(X_test).ravel()
y_pred[:10]

array([ 0.01128928,  0.02023643,  0.02076481,  0.05511703, -0.00446579,
       -0.00581492,  0.04807789,  0.02640491, -0.03554866, -0.01281616])

## Evaluate

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

MAE  : 0.0270
MSE  : 0.0012
RMSE : 0.0346
R2   : 0.3742


## Validate Against scikit-learn

If the from-scratch Normal Equation is implemented correctly, its coefficients and R²
should match `LinearRegression` almost exactly.

In [ ]:
sk_model = LinearRegression()
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_test)

print("Scratch  R2:", round(r2_score(y_test, y_pred), 4))
print("sklearn  R2:", round(r2_score(y_test, sk_pred), 4))
print()
print("Scratch theta (bias + coefs):", model.theta.ravel())
print("sklearn intercept + coefs   :", sk_model.intercept_, list(sk_model.coef_))

Scratch  R2: 0.3742
sklearn  R2: 0.3742

Scratch theta (bias + coefs): [8.71750557e-06 2.33969630e-01 1.41966285e-01 1.62113843e-01
 1.97933031e-01]
sklearn intercept + coefs   : 8.717505570390123e-06 [np.float64(0.23396962972310792), np.float64(0.14196628456535337), np.float64(0.16211384285400368), np.float64(0.1979330307289487)]


## Conclusion

| Model | Features | R² |
|---|---|---|
| SLR (previous project) | BMI only | 0.233 |
| **MLR (this project)** | **bmi, triglycerides, glucose, age** | **0.374** |

Adding three real, non-leaky features on top of BMI improved R² from 0.233 to 0.374 —
a meaningful jump, and confirmation that blood pressure here is explained by a
combination of factors rather than any single one. The scratch Normal Equation
implementation matches scikit-learn's `LinearRegression` on both R² and coefficients,
confirming it's implemented correctly.